# E2.6 · Incident and disclosure obligations

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.5 · Privacy and data protection](https://spbreed.github.io/cyber-commons/lessons/E2.5.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Draft the notification for an agentic incident.

**Why a security engineer needs it.** Materiality assessed for an autonomous actor with a human-actor playbook. The control it builds is: coordinate with D2 in hour one.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The question "is this reportable" has to be answerable in hours, by someone who is already busy. Trigger criteria written during an incident are written under the worst conditions available.

> **At CyberTravels.** Is the $5,000 refund incident reportable, to whom, and by when? That question gets asked at 2am by someone already busy.

## 2 · The framework

```
   the question, asked at 2am, by someone already busy

   is it reportable?  --> to whom?  --> by when?

   +---------------------------------------------+
   | trigger criteria, written in advance:       |
   | data class . autonomy . harm . jurisdiction |
   +---------------------------------------------+

   criteria written during an incident are written badly
```

Incident and disclosure obligations meet agentic incidents badly, for one
specific reason: **broken attribution consumes the clock.**

The clock starts at *awareness* — when you know a reportable event may have
occurred. It does not pause while you work out who did it. So if your logs
attribute an agent's actions to the human whose credential it borrowed (D2.1),
the days you spend establishing what actually happened are deadline days.

Two consequences worth internalising:

1. **Containing fast does not buy reporting time.** You can contain in an hour
   and still miss a 72-hour deadline.
2. **You will have to disclose before attribution is complete.** So the sentence
   you send when you know an agent acted but cannot yet say which one needs to
   be drafted *now*, not during the incident.

## 3 · The procedure, as a skill

Containment takes an hour and establishing who acted takes forty-eight, on a seventy-two hour clock. The skill breaks the deadline into phases, finds the dominant one, and re-runs it with delegation chains recorded.

In [ ]:
# skills/regulatory/disclosure-phase-breakdown/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: disclosure-phase-breakdown
description: >-
  Break a disclosure deadline into its phases and find the one that consumes
  most of it — usually establishing who acted rather than containment. Use when
  a reporting obligation is being planned for, or was missed.
allowed-tools: Read, Grep, Glob
---

# Two-thirds of the clock is establishing who acted

Disclosure planning concentrates on containment because that is the exciting
part. The phase breakdown says otherwise: containment in an hour, and
establishing the acting identity in forty-eight, on a seventy-two hour clock.
The delay is attribution, and attribution is fixed months earlier by what the
logs record.

## When to use this

Planning for a reporting obligation, and at post-incident review when a deadline
was met late or narrowly.

## Procedure

**1 — Enumerate the phases.** Detection, triage, containment, establishing who
acted, scoping affected subjects, drafting, approval, submission. All of them,
including the approval step everyone forgets.

**2 — Attribute hours to each from a real incident** or a tabletop. Estimates
here are systematically optimistic; use measured values where you have them and
mark the rest as estimates.

**3 — Find the dominant phase.** It is usually attribution or scoping. Report it
as a proportion of the whole clock, which is what makes it actionable.

**4 — Re-run with delegation chains recorded.** Model what attribution costs
when the acting identity and the chain are on every record. The difference is
the business case for A2.7-style attribution, in hours against a regulatory
deadline.

**5 — Pre-draft what can be pre-drafted.** The regime, the template, the
distribution list, the approver. Drafting under time pressure is where the
avoidable hours are.

## Output contract

```json
{
  "obligation": {"regime": "str", "hours": 0},
  "phases": [{"phase": "str", "hours": 0, "measured": false}],
  "total_hours": 0,
  "dominant": {"phase": "str", "share": 0.0},
  "with_attribution": {"phase_hours": 0, "total_hours": 0, "meets_deadline": true},
  "pre_drafted": ["str"]
}
```

## Failure modes

- **Optimising containment.** It is already the fast phase.
- **Estimating attribution.** Measure it once; it is worse than you think.
- **Omitting approval.** It is a real phase with a real queue.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/regulatory/disclosure-phase-breakdown/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/regulatory/disclosure-phase-breakdown/scripts/disclosure_phase_breakdown.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Break the disclosure clock into phases and find the one that consumes most of it.

This is the executable half of the `disclosure-phase-breakdown` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
t0 = time.time(); H = 3600

def clock(awareness, containment, report, deadline_hours):
    to_contain = (containment - awareness)/H
    to_report  = (report - awareness)/H
    return {"contain_h": round(to_contain,1), "report_h": round(to_report,1),
            "deadline": deadline_hours, "met": to_report <= deadline_hours,
            "margin_h": round(deadline_hours - to_report, 1)}

SCENARIOS = {
 "attribution sound":            (t0 + 2*H,  t0 + 20*H),
 "attribution broken, 3d scope": (t0 + 6*H,  t0 + 92*H),
 "fast containment, slow scope": (t0 + 1*H,  t0 + 80*H),
}
print(f"{'scenario':32s}{'contain':>9}{'report':>9}{'met':>6}{'margin':>9}")
print("-" * 66)
for name, (c, r) in SCENARIOS.items():
    k = clock(t0, c, r, 72)
    print(f"{name:32s}{k['contain_h']:>9.1f}{k['report_h']:>9.1f}"
          f"{str(k['met']):>6}{k['margin_h']:>9.1f}")
print("\nThe third row contained in ONE HOUR and missed by 8 hours.")

# Where the time actually goes when attribution is broken.
PHASES = [
 ("alert fires → analyst picks it up",        3,  "queue depth"),
 ("confirm an incident",                      6,  "is this real?"),
 ("establish WHO acted",                     48,  "logs name the human; agents hidden"),
 ("scope what was touched",                  24,  "must walk the delegation chain (D2.3)"),
 ("legal determines reportability",            8,  "needs the scope"),
 ("draft and send",                            3,  ""),
]
elapsed = 0
print(f"{'phase':38s}{'hours':>7}{'cumulative':>12}  note")
print("-" * 82)
for name, h, note in PHASES:
    elapsed += h
    flag = "  ← DEADLINE PASSED" if elapsed > 72 else ""
    print(f"{name:38s}{h:>7}{elapsed:>12}{flag}  {note}")
print(f"\ntotal {elapsed}h against a 72h deadline")
attribution_cost = PHASES[2][1]
print(f"the attribution phase alone is {attribution_cost}h — "
      f"{attribution_cost/72:.0%} of the entire deadline")
assert elapsed > 72

def with_act_chains(phases):
    """With an acting-identity field, 'who acted' is a query, not an investigation."""
    return [(n, (0.5 if n.startswith("establish WHO") else h), note)
            for n, h, note in phases]

fixed = with_act_chains(PHASES)
total_fixed = sum(h for _, h, _ in fixed)
print(f"with act chains recorded (A2.5 + EV-1): {total_fixed}h vs {elapsed}h")
print(f"deadline met: {total_fixed <= 72}")
assert total_fixed <= 72

PRE_DRAFTED = """
We are notifying you of an incident under [instrument], first identified at
[awareness timestamp].

An automated system operating within our environment performed actions that may
have affected [scope]. Our logging currently attributes these actions to the
authenticated principal on whose behalf the system was acting; we are working to
establish which specific automated component performed them.

Containment: [action] completed at [time].
We will provide an update within [period], including the completed attribution.
"""
print("\nPRE-DRAFTED DISCLOSURE (write this now, not during the incident):")
print(PRE_DRAFTED)
print("It is honest, it starts the notification, and it does not claim an")
print("attribution you cannot yet support.")

# Verify: the runbook needs two owners, not one.
def runbook_check(containment_owner, disclosure_owner, clock_starts_at,
                  has_predrafted):
    problems = []
    if containment_owner == disclosure_owner:
        problems.append("one owner for both workstreams — they compete under time pressure")
    if clock_starts_at != "awareness":
        problems.append(f"clock starts at {clock_starts_at!r}; a regulator will use awareness")
    if not has_predrafted:
        problems.append("no pre-drafted disclosure for incomplete attribution")
    return (not problems), problems

for label, args in (("as usually written", ("IR lead", "IR lead", "confirmation", False)),
                    ("corrected", ("IR lead", "legal/compliance lead", "awareness", True))):
    ok, problems = runbook_check(*args)
    print(f"{label:22s} sound={ok}")
    for p in problems: print(f"   ⚠ {p}")
assert runbook_check("IR lead", "legal/compliance lead", "awareness", True)[0]

## What you just proved

One-hour containment still misses the 72-hour deadline when scoping is slow. The phase breakdown totals 92 hours, of which establishing who acted is 48 — two-thirds of the entire deadline. Recording act chains cuts the total to 44.5 hours and meets the deadline. The runbook check flags a shared owner, a late clock start and a missing pre-drafted disclosure.

## Your turn

Draft the disclosure sentence you would send when you know an agent acted but cannot say which one. Getting legal to agree that wording takes weeks in peacetime and is impossible at hour 60.

---

**Next → [E2.7 · Documentation that survives supervision](https://spbreed.github.io/cyber-commons/lessons/E2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*